# G06 — Positive-pool size ablation of G05 using the same trained model

Generator **G06 (`06_ldm_extra1361_fromscratch`) is not a separately trained model**. It inherits G05's frozen validation selection and repeats generation/filtering under a larger raw-pool target: 4,083 positive samples rather than 2,722, while retaining 1,361 filtered positives. Its valid scientific interpretation is therefore a **pool-size and filtering ablation of the G05 model**, not an independent architecture, training run, or source of model-level replication.

G06 keeps its generated images, logs, pool metrics, and figures under experiment-specific paths so that the intervention can be audited without overwriting G05. The shared model identity must nevertheless remain explicit in registries, benchmark tables, and prose. Any apparent G05–G06 difference can arise from the sampled pool or filtering; it cannot be attributed to training or a second checkpoint-selection event.

Like G05, the executable generation path in this notebook is positive-class focused. A later cell can characterize an already available negative filtered directory, but this notebook does not construct an independent two-class model or a complete negative generation workflow. G06 is descriptive and is not the selected from-scratch generator for downstream classifier augmentation.


## 1. Runtime bootstrap and accelerator discovery

The bootstrap resolves the repository and utility paths, locates XLA `libdevice`, prints the NVIDIA inventory when available, and defines separate child-process policies for training-style commands and parallel generation. A portable execution should obtain device visibility from the environment or runtime discovery rather than a notebook-specific hardware identifier.

Hardware diagnostics are recorded for troubleshooting only. They do not distinguish G06 from G05, because both experiment labels ultimately reference the same trained checkpoint family.


In [ ]:
# === Bootstrap unificato notebooks/ ===
# Funziona dalla root del progetto e da ogni sottocartella della struttura notebooks/.
import sys as _sys
from pathlib import Path as _Path

def _find_mammo_root():
    for _candidate in [_Path.cwd().resolve(), *_Path.cwd().resolve().parents]:
        if _candidate.name == "MammoDiffusion":
            return _candidate
        if (_candidate / "data").is_dir() and (_candidate / "notebooks").is_dir():
            return _candidate
    raise FileNotFoundError("Root MammoDiffusion non trovata da " + str(_Path.cwd()))

PROJECT_ROOT = _find_mammo_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
UTILITY_DIR = NOTEBOOKS_DIR / "utility"
for _path in (str(UTILITY_DIR), str(NOTEBOOKS_DIR)):
    if _path not in _sys.path:
        _sys.path.insert(0, _path)

# === Fine bootstrap unificato ===

import os
import subprocess
import sys
from pathlib import Path

CUDA_ROOT = Path(os.environ.get("MAMMODIFFUSION_CUDA_ROOT", os.environ.get("CONDA_PREFIX", sys.prefix)))

libdevice_path = CUDA_ROOT / "nvvm" / "libdevice" / "libdevice.10.bc"
if libdevice_path.exists():
    os.environ["XLA_FLAGS"] = f"--xla_gpu_cuda_data_dir={CUDA_ROOT}"
else:
    print("libdevice.10.bc non trovato in:", libdevice_path)

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.stdout.strip():
        print("GPU fisiche disponibili:")
        print(result.stdout.strip())
except FileNotFoundError:
    print("nvidia-smi non disponibile in questo ambiente.")

print("XLA_FLAGS:", os.environ.get("XLA_FLAGS", ""))
# Inherit CUDA visibility by default; optionally override it for training subprocesses.
TRAIN_GPU_VISIBLE_DEVICES = os.environ.get("MAMMODIFFUSION_TRAIN_GPU")

def training_subprocess_env():
    env = os.environ.copy()
    if TRAIN_GPU_VISIBLE_DEVICES is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(TRAIN_GPU_VISIBLE_DEVICES)
    return env

# Generazione multi-GPU (non influenza il training).
PARALLEL_GENERATION = True
GENERATION_GPU_DEVICES = "auto"
GENERATION_MAX_WORKERS = None

def add_generation_parallel_args(command):
    if PARALLEL_GENERATION:
        command.extend(["--generation-gpus", GENERATION_GPU_DEVICES])
        if GENERATION_MAX_WORKERS is not None:
            command.extend(["--max-generation-workers", str(GENERATION_MAX_WORKERS)])
    else:
        command.extend(["--generation-gpus", "off"])
    return command

## 2. Controlled dependency setup

Dependency installation is disabled by default through `INSTALL_DEPENDENCIES=False`. When intentionally enabled in a new environment, the cell installs the numerical, TensorFlow, PyTorch, and generative-metric packages required by the helpers. It produces no model artifact and should not be toggled during a scientific rerun without recording the resolved versions, because package changes can alter preprocessing, sampling, or metric implementations.


In [ ]:
# L'ambiente tf-gpu del progetto include gia' le dipendenze.
# Impostare True solo quando si prepara intenzionalmente un ambiente nuovo.
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    %pip install -q --upgrade pip
    %pip install -q pandas numpy matplotlib scikit-learn pillow gdown tensorflow scikit-image scipy psutil codecarbon torch torchvision torchmetrics torch-fidelity prdc
else:
    print('Dipendenze gia presenti: installazione saltata.')

## 3. G06 paths, shared G05 checkpoints, and phase flags

The setup creates G06-specific experiment/result directories but assigns `CHECKPOINTS_DIR` to `experiments/diffusers/05_ldm_basic_fromscratch/checkpoints_ldm`. This explicit cross-reference records the essential source relationship: G06 owns a different generated pool and reporting directory, not different learned weights.

`RUN_TRAINING_PHASE = False` prevents duplicate training, and `RUN_EVALUATION_PHASE = False` preserves G05 as the sole checkpoint-selection owner while delegating to explicit selection-file and checkpoint checks. Generation and filtering run only when their own flags are set, so an unapproved full regeneration cannot start. Non-zero subprocess exits raise errors and logs remain under the G06 experiment. A publication rerun should set only the flags it intends and verify both the inherited G05 checkpoint identity and the G06 pool manifest.


In [ ]:
from pathlib import Path
import shutil
import sys

PROJECT_NAME = "MammoDiffusion"
EXPERIMENT_NAME = "diffusers/06_ldm_extra1361_fromscratch"
RESULTS_STAGE_NAME = "2_diffusers/06_ldm_extra1361_fromscratch"

# Se serve su Colab/Drive:
# PROJECT_ROOT_OVERRIDE = Path("/content/drive/MyDrive/MammoDiffusion")
PROJECT_ROOT_OVERRIDE = None

def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
    if override is not None:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE non esiste: {root}")
        return root

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == project_name:
            return candidate
        has_notebooks = (candidate / "notebooks").exists() or (candidate / "notebooks").exists()
        if ((candidate / "data").exists() and has_notebooks) or ((candidate / ".git").exists() and has_notebooks):
            return candidate

    for candidate in [
        cwd / project_name,
        Path("/content") / project_name,
        Path("/content/drive/MyDrive") / project_name,
        Path.home() / project_name,
    ]:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        "Non riesco a trovare la root MammoDiffusion. "
        "Esegui il notebook dalla repo o imposta PROJECT_ROOT_OVERRIDE."
    )

PROJECT_ROOT = find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / ("notebooks" if (PROJECT_ROOT / "notebooks").is_dir() else "notebooks")
UTILITY_DIR = NOTEBOOKS_DIR / "utility" if (NOTEBOOKS_DIR / "utility").is_dir() else NOTEBOOKS_DIR
DATA_DIR = PROJECT_ROOT / "data"
DATA_PROCESSED_DIR = DATA_DIR / "processed"
ARCHIVES_DIR = DATA_DIR / "archives"
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / EXPERIMENT_NAME
ACTIVE_G06_FILTERED_DIR = EXPERIMENT_DIR / "synthetic_filtered_positive"
SOURCE_EXPERIMENT_DIR = PROJECT_ROOT / "experiments/diffusers/05_ldm_basic_fromscratch"

MODELS_DIR = EXPERIMENT_DIR / "models"
CHECKPOINTS_DIR = SOURCE_EXPERIMENT_DIR / "checkpoints_ldm"
LATENTS_DIR = EXPERIMENT_DIR / "latents"
LOGS_DIR = EXPERIMENT_DIR / "logs"
RESULTS_DIR = PROJECT_ROOT / "results" / RESULTS_STAGE_NAME
RESULTS_PLOTS_DIR = RESULTS_DIR / "plots"
RESULTS_METRICS_DIR = RESULTS_DIR / "metrics"
RESULTS_ECOTRACKER_DIR = RESULTS_DIR / "ecotracker"
HELPER_PATH = UTILITY_DIR / "train_ldm.py"
VAE_HELPER_PATH = UTILITY_DIR / "train_vae.py"

for directory in [
    DATA_PROCESSED_DIR,
    ARCHIVES_DIR,
    EXPERIMENT_DIR,
    MODELS_DIR,
    LATENTS_DIR,
    LOGS_DIR,
    RESULTS_DIR,
    RESULTS_PLOTS_DIR,
    RESULTS_METRICS_DIR,
    RESULTS_ECOTRACKER_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PROCESSED_DIR:", DATA_PROCESSED_DIR)
print("EXPERIMENT_DIR:", EXPERIMENT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("HELPER_PATH:", HELPER_PATH)
print("VAE_HELPER_PATH:", VAE_HELPER_PATH)

if not HELPER_PATH.exists():
    raise FileNotFoundError(f"Helper non trovato: {HELPER_PATH}")
if not VAE_HELPER_PATH.exists():
    raise FileNotFoundError(f"Helper VAE non trovato: {VAE_HELPER_PATH}")

In [ ]:
# EXPLICIT_PHASE_FLAGS_V1
# One boolean per phase. Every flag is False, so an ordinary Run All reads the
# artifacts already on disk and reports them: it never retrains and never
# regenerates. To do real work, set the flags for the phases you intend to run
# and execute the notebook top to bottom. Flags are independent -- filtering can
# be redone without regenerating the pool it selects from.
#
#   RUN_TRAINING_PHASE    train the model (hours of GPU)
#   RUN_GENERATION_PHASE  sample a full RAW image pool from the selected checkpoint
#   RUN_EVALUATION_PHASE  score checkpoints and record the selection
#   RUN_FILTER_PHASE      re-run the adaptive filter over the existing RAW pool
#   RUN_VALIDATION_PHASE  RAW-vs-filtered comparison; keeps its own content-aware cache
#
# Leaving a flag False asserts that the phase's artifact is already complete.
# The cells below check that claim and raise if it does not hold, rather than
# reporting a number they did not verify.
RUN_TRAINING_PHASE = False  # 06 reuses the checkpoint trained by 05
RUN_GENERATION_PHASE = False
RUN_EVALUATION_PHASE = False  # G05 stays the only owner of checkpoint selection
RUN_FILTER_PHASE = False
RUN_VALIDATION_PHASE = False

## 4. Shared preprocessed dataset

This section performs the same structural audit used by G05: both class directories must be populated for every split, and the four metadata CSV files must be available. Existing complete data are reused; otherwise the shared processed archive is downloaded, validated, extracted, and checked before execution continues.

The dataset is common to G05 and G06. Reconstructing metadata from filenames is a recovery mechanism, not a new data split, and it cannot create independent evidence. Test data remain reserved for final classifier evaluation.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

import pandas as pd

try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown

# ZIP condiviso: train/0, train/1, val/0, val/1, test/0, test/1, metadata/*.csv
PROCESSED_DRIVE_ID = "1qQral_BIBlMl0QN3PllJukdYTOmNGWr3"
PROCESSED_ZIP_PATH = ARCHIVES_DIR / "processed.zip"
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
EXPECTED_SPLITS = ["train", "val"]
EXPECTED_LABELS = ["0", "1"]
REQUIRED_METADATA = ["all_processed.csv", "train.csv", "val.csv"]

def count_images(folder):
    folder = Path(folder)
    if not folder.is_dir():
        return 0
    return sum(
        1
        for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )

def get_split_label_counts(processed_dir):
    processed_dir = Path(processed_dir).resolve()
    rows = []
    for split in EXPECTED_SPLITS:
        for label in EXPECTED_LABELS:
            rows.append(
                {
                    "split": split,
                    "label": int(label),
                    "folder": str(processed_dir / split / label),
                    "n_images": count_images(processed_dir / split / label),
                }
            )
    return pd.DataFrame(rows)

def processed_dataset_ready(processed_dir):
    processed_dir = Path(processed_dir).resolve()
    if not processed_dir.exists():
        return False
    counts_df = get_split_label_counts(processed_dir)
    ready = (counts_df["n_images"] > 0).all()
    if not ready and any((processed_dir / split).exists() for split in EXPECTED_SPLITS):
        print("Dataset preprocessato trovato, ma struttura incompleta:")
        print(counts_df[["split", "label", "n_images"]].to_string(index=False))
    return bool(ready)

def metadata_complete(processed_dir):
    metadata_dir = Path(processed_dir) / "metadata"
    return all((metadata_dir / name).exists() for name in REQUIRED_METADATA)

def parse_filename_metadata(image_path):
    parts = Path(image_path).stem.split("_")
    patient_id = parts[0] if len(parts) >= 1 else Path(image_path).stem
    image_id = parts[1] if len(parts) >= 2 else Path(image_path).stem
    laterality = parts[2] if len(parts) >= 3 else "unknown"
    view = parts[3] if len(parts) >= 4 else "unknown"
    return patient_id, image_id, laterality, view

def rebuild_metadata_from_folders(processed_dir):
    processed_dir = Path(processed_dir).resolve()
    metadata_dir = processed_dir / "metadata"
    metadata_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for split in EXPECTED_SPLITS:
        for label_name in EXPECTED_LABELS:
            label_dir = processed_dir / split / label_name
            label = int(label_name)
            for image_path in sorted(label_dir.rglob("*")):
                if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                    continue
                patient_id, image_id, laterality, view = parse_filename_metadata(image_path)
                rows.append(
                    {
                        "patient_id": str(patient_id),
                        "image_id": str(image_id),
                        "laterality": str(laterality),
                        "view": str(view),
                        "label": label,
                        "cancer": label,
                        "patient_label": label,
                        "split": split,
                        "source": "real",
                        "original_path": "",
                        "processed_path": str(image_path),
                    }
                )
    if not rows:
        raise FileNotFoundError(f"Nessuna immagine valida trovata in {processed_dir}")

    df = pd.DataFrame(rows).sort_values(
        ["split", "label", "patient_id", "image_id"]
    ).reset_index(drop=True)
    df.to_csv(metadata_dir / "all_processed.csv", index=False)
    for split in EXPECTED_SPLITS:
        df[df["split"] == split].reset_index(drop=True).to_csv(
            metadata_dir / f"{split}.csv",
            index=False,
        )
    print("Metadata CSV ricostruiti in:", metadata_dir)
    print(pd.crosstab(df["split"], df["label"]))

def ensure_metadata(processed_dir):
    if metadata_complete(processed_dir):
        print("Metadata CSV gia' presenti:", Path(processed_dir) / "metadata")
        return
    print("Metadata CSV mancanti: li ricostruisco da train/val/test/0-1.")
    rebuild_metadata_from_folders(processed_dir)

def download_processed_zip():
    if PROCESSED_ZIP_PATH.exists():
        if zipfile.is_zipfile(PROCESSED_ZIP_PATH):
            print("Archivio gia' presente, salto download:", PROCESSED_ZIP_PATH)
            return
        print("Archivio presente ma non valido: lo riscarico.")
        PROCESSED_ZIP_PATH.unlink()
    print("Download di processed.zip da Google Drive...")
    gdown.download(id=PROCESSED_DRIVE_ID, output=str(PROCESSED_ZIP_PATH), quiet=False)

    if not PROCESSED_ZIP_PATH.exists() or PROCESSED_ZIP_PATH.stat().st_size == 0:
        raise RuntimeError("Download fallito. Controlla la condivisione del file Drive.")
    if not zipfile.is_zipfile(PROCESSED_ZIP_PATH):
        raise RuntimeError(f"Il file scaricato non e' uno ZIP valido: {PROCESSED_ZIP_PATH}")
    print("Download completato:", PROCESSED_ZIP_PATH)

def clear_processed_dir():
    DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    for item in DATA_PROCESSED_DIR.iterdir():
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()

def extract_processed_zip():
    print("Estrazione di processed.zip dentro:", DATA_PROCESSED_DIR)
    clear_processed_dir()
    with zipfile.ZipFile(PROCESSED_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(DATA_PROCESSED_DIR)

def prepare_processed_dataset():
    if processed_dataset_ready(DATA_PROCESSED_DIR):
        print("Dataset preprocessato gia' presente: salto download ed estrazione.")
        ensure_metadata(DATA_PROCESSED_DIR)
        return DATA_PROCESSED_DIR.resolve()

    PROCESSED_ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
    download_processed_zip()
    extract_processed_zip()

    if not processed_dataset_ready(DATA_PROCESSED_DIR):
        counts_df = get_split_label_counts(DATA_PROCESSED_DIR)
        print(counts_df[["split", "label", "folder", "n_images"]].to_string(index=False))
        raise FileNotFoundError(
            "processed.zip estratto, ma la struttura non e' quella attesa. "
            "Lo ZIP deve contenere direttamente train/0, train/1, val/0, val/1, "
            "test/0, test/1 e metadata/*.csv."
        )

    ensure_metadata(DATA_PROCESSED_DIR)
    return DATA_PROCESSED_DIR.resolve()

DATASET_ROOT = prepare_processed_dataset()
print("DATASET_ROOT:", DATASET_ROOT)
print(get_split_label_counts(DATASET_ROOT)[["split", "label", "n_images"]].to_string(index=False))

## 5. Shared-model preflight — no VAE training

This cell does not fit or copy a VAE. It verifies that the shared G05 checkpoint directory contains at least one LDM Keras model and fails with an explicit instruction to execute G05 first when the dependency is absent. The check prevents G06 from silently substituting randomly initialized or unrelated weights.

Because the learned representation is inherited, any VAE or U-Net training plots shown below are contextual diagnostics only. They must not be described as G06 training outcomes.


In [ ]:
# Nessun training duplicato: 06 estende il pool generato dal modello di 05.
if not any(CHECKPOINTS_DIR.glob('ldm_unet_*.keras')):
    raise FileNotFoundError(
        f"Checkpoint condivisi non trovati: {CHECKPOINTS_DIR}. "
        "Esegui prima 05_LDM_Basic_FromScratch.ipynb."
    )
print("Checkpoint condivisi da 05:", CHECKPOINTS_DIR)

### 5.1 Read-only inherited VAE diagnostics

The next cell displays any VAE metric and reconstruction panels already present in the G06 results namespace, cropping the reconstruction montage for readability. It never launches training and tolerates absent plots by reporting them.

These figures can document the inherited latent representation, but they do not demonstrate a second VAE fit. The model identity remains G05 regardless of whether a copied diagnostic image exists.


In [ ]:
from IPython.display import display
from PIL import Image as PILImage

for plot_name in ["vae_metrics.png", "vae_reconstruction.png"]:
    plot_path = RESULTS_PLOTS_DIR / plot_name
    if plot_path.exists():
        img = PILImage.open(plot_path)
        if plot_name == "vae_reconstruction.png":
            # Mostra solo le prime 3 ricostruzioni (la griglia salvata ha 6 colonne)
            w, h = img.size
            img = img.crop((0, 0, w // 2, h))
        display(img)
        print("Plot:", plot_path)
    else:
        print("Plot non trovato:", plot_path)

## 6. Shared LDM checkpoint preflight — training intentionally disabled

A second explicit guard confirms that G05 LDM checkpoints remain available before downstream work begins. There is no call to `train_ldm.py`; `RUN_TRAINING_PHASE = False` and the shared path make G06 a generation-pool intervention by construction.

This safeguard avoids accidental retraining under the G06 label. It also means G06 cannot be counted as an independent seed, replicate, or architecture in statistical comparisons.


In [ ]:
# Nessun training duplicato: 06 estende il pool generato dal modello di 05.
if not any(CHECKPOINTS_DIR.glob('ldm_unet_*.keras')):
    raise FileNotFoundError(
        f"Checkpoint condivisi non trovati: {CHECKPOINTS_DIR}. "
        "Esegui prima 05_LDM_Basic_FromScratch.ipynb."
    )
print("Checkpoint condivisi da 05:", CHECKPOINTS_DIR)

### 6.1 Read-only inherited LDM training curve

The cell displays `ldm_metrics.png` if it is available and otherwise reports its absence. No checkpoint is loaded or updated. The curve provides context for the shared G05 trajectory; it is not evidence of G06-specific optimization.


In [ ]:
from IPython.display import display
from PIL import Image as PILImage

plot_path = RESULTS_PLOTS_DIR / "ldm_metrics.png"
if plot_path.exists():
    display(PILImage.open(plot_path))
    print("Plot:", plot_path)
else:
    print("Plot non trovato:", plot_path)

## 7. Inherited G05 validation selection

G06 is a candidate-pool ablation, so it must not reselect or overwrite a model. The code reads G05's frozen `evaluation/best_checkpoint.json`, reroots the recorded filename through the current project directory, and requires the selected periodic checkpoint to exist. `RUN_EVALUATION_PHASE = False` makes this boundary explicit.

No G06 evaluation subprocess is launched and no selected-model copy is written into the shared G05 checkpoint directory. The G05 sweep remains the sole validation-selection event; G06 changes only the number of raw candidates presented to the downstream deterministic filter.


### 7.1 Frozen positive-class checkpoint rule

G05 selected its checkpoint by minimizing positive-class FID and using positive-class Inception Score only as a tie-breaker; label-0 metrics were auxiliary. G06 consumes that recorded decision without screening the trajectory again.

Freezing the periodic checkpoint isolates the intended pool-size intervention more cleanly. G06 is therefore not an “extra1361 model”: it is an additional positive-sample pool generated from the same learned weights and sampling configuration used by G05.


In [ ]:
# IDEMPOTENT_GUARD_V1:evaluation
# Keep configuration available to downstream cells even when this phase is skipped.
import json

EVAL_INCEPTION_BATCH = 8

G05_SELECTION_PATH = SOURCE_EXPERIMENT_DIR / "evaluation" / "best_checkpoint.json"
if not G05_SELECTION_PATH.is_file():
    raise FileNotFoundError(f"Frozen G05 selection is unavailable: {G05_SELECTION_PATH}")

with G05_SELECTION_PATH.open(encoding="utf-8") as handle:
    G05_SELECTION = json.load(handle)

recorded_checkpoint = Path(G05_SELECTION["best_checkpoint"])
G05_SELECTED_CHECKPOINT = CHECKPOINTS_DIR / recorded_checkpoint.name
if not G05_SELECTED_CHECKPOINT.is_file() or G05_SELECTED_CHECKPOINT.stat().st_size == 0:
    raise FileNotFoundError(
        f"Frozen G05 periodic checkpoint is unavailable: {G05_SELECTED_CHECKPOINT}"
    )
print("Inherited G05 selection:", G05_SELECTION["best_checkpoint_id"])
print("Portable checkpoint path:", G05_SELECTED_CHECKPOINT)

### 7.2 PRDC trajectories for the shared checkpoints

This read-only plot loads the original G05 sweep table, shows precision, recall, density, and coverage for both labels, marks the frozen validation-selected checkpoint, and saves a copy of the diagnostic as `checkpoint_prdc_comparison.png` in the G06 result namespace.

The figure can reveal sampling instability or fidelity–coverage trade-offs, but it does not alter the primary positive-FID rule and cannot support a claim of distinct G06 learning dynamics.


In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sweep_df = (
    pd.read_csv(SOURCE_EXPERIMENT_DIR / "evaluation" / "sweep_results.csv")
    .sort_values("checkpoint_order")
    .reset_index(drop=True)
)
with open(G05_SELECTION_PATH, encoding="utf-8") as handle:
    best_ckpt = json.load(handle)

best_id = best_ckpt["best_checkpoint_id"]
x = np.arange(len(sweep_df))
labels = sweep_df["checkpoint_id"].astype(str).tolist()
best_positions = sweep_df.index[sweep_df["checkpoint_id"].astype(str) == best_id].tolist()
best_x = best_positions[0] if best_positions else None

metric_specs = [
    ("precision_0", "precision_1", "Precision"),
    ("recall_0", "recall_1", "Recall"),
    ("density_0", "density_1", "Density"),
    ("coverage_0", "coverage_1", "Coverage"),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
for ax, (col_0, col_1, title) in zip(axes.ravel(), metric_specs):
    ax.plot(x, sweep_df[col_0].astype(float), marker="o", linewidth=1.8, label="classe 0")
    ax.plot(x, sweep_df[col_1].astype(float), marker="s", linewidth=1.8, label="classe 1")
    if best_x is not None:
        ax.axvline(best_x, color="crimson", linestyle="--", linewidth=1.4)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="best")

fig.suptitle(f"Trend PRDC per checkpoint lungo lo sweep - BEST: {best_id}", fontsize=14)
output_path = RESULTS_PLOTS_DIR / "checkpoint_prdc_comparison.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()

print("Salvato:", output_path)

## 8. Positive-pool expansion and adaptive filtering

The generation helper uses the shared validation-selected G05 checkpoint and targets 4,083 raw positive images—conceptually the original 2,722-image pool plus 1,361 additional samples—before retaining 1,361 filtered positives in G06's registered experiment-local directory. A generation repair uses `--mode both`; a filter-only repair enters `--mode filter` directly. Existing readable indices are reused, corrupt or missing images are regenerated, and multi-GPU reservations prevent duplicated work.

G06-specific raw images, filter manifests, reports, figures, logs, and energy events remain separate from G05. The intervention tests how the candidate pool available to the same deterministic filter affects the retained set. It does not test a new generator, and the resulting filtered positives must be labeled as a pool-size ablation in the unified benchmark.


In [ ]:
# IDEMPOTENT_GUARD_V1:generation
# Keep configuration available to validation/final-evaluation cells when generation is skipped.
GEN_MODE = None  # risolto dai flag di fase piu' sotto
GEN_N_RAW = 4083  # 2722 gia' presenti dal primo esperimento + 1361 nuove raw
GEN_N_SELECTED = 1361
GEN_TARGET_LABEL = 1
GEN_BATCH_SIZE = 1
GEN_SAMPLE_STEPS = 100
GEN_GUIDANCE_SCALE = 1.5
GEN_MODEL_PATH = G05_SELECTED_CHECKPOINT
GEN_ECO_TRACK = True

if RUN_GENERATION_PHASE or RUN_FILTER_PHASE:
    GEN_MODE = "both" if RUN_GENERATION_PHASE else "filter"
    import os
    import subprocess
    import time

    gen_log_path = LOGS_DIR / "ldm_generate.log"
    gen_cmd = [
        sys.executable,
        str(UTILITY_DIR / "generate_ldm.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--model-path", str(GEN_MODEL_PATH),
        "--mode", GEN_MODE,
        "--n-raw", str(GEN_N_RAW),
        "--n-selected", str(GEN_N_SELECTED),
        "--target-label", str(GEN_TARGET_LABEL),
        "--filtered-dir", str(ACTIVE_G06_FILTERED_DIR),
        "--batch-size", str(GEN_BATCH_SIZE),
        "--sample-steps", str(GEN_SAMPLE_STEPS),
        "--guidance-scale", str(GEN_GUIDANCE_SCALE),
        "--results-stage-name", RESULTS_STAGE_NAME,
    ]

    if GEN_ECO_TRACK:
        gen_cmd.append("--eco-track")

    add_generation_parallel_args(gen_cmd)
    env = os.environ.copy()
    print("Comando generation:")
    print(" ".join(gen_cmd))
    print("Log:", gen_log_path)
    print("CUDA_VISIBLE_DEVICES:", env.get("CUDA_VISIBLE_DEVICES", ""))
    print("XLA_FLAGS:", env.get("XLA_FLAGS", ""))

    with open(gen_log_path, "w", encoding="utf-8") as log_file:
        proc_gen = subprocess.Popen(
            gen_cmd,
            cwd=str(PROJECT_ROOT),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )

    pid_gen = proc_gen.pid
    print(f"Generation avviata - PID {pid_gen}")

    with open(gen_log_path, "r", encoding="utf-8", errors="replace") as log_file:
        while proc_gen.poll() is None:
            line = log_file.readline()
            if line:
                print(line, end="", flush=True)
            else:
                time.sleep(0.5)

        for line in log_file:
            print(line, end="", flush=True)

    print("Processo generation terminato con return code:", proc_gen.returncode)
    if proc_gen.returncode != 0:
        raise RuntimeError(f"Generation fallita. Controlla il log: {gen_log_path}")

### 8.1 Filter rejection profile

The cell reads `synthetic_filter_summary.json`, counts each rejection category, and saves `synthetic_filter_reject_reasons.png`. These counts make the effect of enlarging the raw pool auditable and can be compared descriptively with G05 under the same filter implementation.

Rejection categories are image-quality heuristics, not clinical labels. Differences from G05 may arise solely because the candidate pool changed and must not be presented as evidence of improved model training.


In [ ]:
import json

import matplotlib.pyplot as plt

with open(RESULTS_METRICS_DIR / "synthetic_filter_summary.json", encoding="utf-8") as handle:
    filter_summary = json.load(handle)

reject_counts = filter_summary["reject_counts"]
n_raw = filter_summary["n_raw"]
n_accepted = filter_summary["n_accepted"]
n_selected = filter_summary["n_selected"]

reasons = list(reject_counts.keys())
counts = [reject_counts[reason] for reason in reasons]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(reasons, counts, color="#c0392b")
ax.bar_label(bars, padding=3)
ax.set_title(f"Motivi di scarto del filtro - {n_raw} raw, {n_accepted} accettate, {n_selected} selezionate")
ax.set_ylabel("Immagini scartate")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()

output_path = RESULTS_PLOTS_DIR / "synthetic_filter_reject_reasons.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()

print("Salvato:", output_path)
print("Conteggi:", reject_counts)

### 8.2 Read-only accepted/rejected examples and score distributions

This section displays the filter sample panel and distribution diagnostics produced by the preceding pipeline. The plots contrast accepted and rejected examples and compare simple foreground/contrast statistics with real references.

They are qualitative audit aids rather than blinded assessments. A larger pool can yield more high-scoring examples by selection alone, so improved-looking panels do not imply a change in the shared G05 generator distribution.


In [ ]:
from IPython.display import display
from PIL import Image as PILImage

for plot_name in ["synthetic_filter_sample.png", "synthetic_filter_distribution.png"]:
    plot_path = RESULTS_PLOTS_DIR / plot_name
    if plot_path.exists():
        display(PILImage.open(plot_path))
        print("Plot:", plot_path)
    else:
        print("Plot non trovato:", plot_path)

### 8.3 Count-matched raw-versus-filtered validation analysis

A separate `generate_ldm.py --mode validate` call compares the complete G06 raw pool, a deterministic seed-42 raw subset with the same size as the retained pool, and the registered experiment-local filtered positives against validation references. The explicit filtered path and content signatures prevent substitution of the distinct shared-data copy. It writes class-scoped raw-versus-filtered CSV/JSON evidence and sustainability logs.

Count matching reduces a direct sample-size confound but does not remove selection bias introduced by choosing the highest-quality images from a larger candidate pool. FID, Inception Score, and PRDC should be interpreted jointly, with particular attention to possible loss of recall or coverage.


In [ ]:
# IDEMPOTENT_GUARD_V1:generation
if RUN_VALIDATION_PHASE:
    import os
    import subprocess
    import time

    VALIDATE_N_RAW = GEN_N_RAW
    VALIDATE_N_SELECTED = GEN_N_SELECTED
    VALIDATE_TARGET_LABEL = GEN_TARGET_LABEL
    VALIDATE_BALANCED_SEED = 42
    VALIDATE_INCEPTION_BATCH = EVAL_INCEPTION_BATCH
    VALIDATE_IS_SPLITS = 10
    VALIDATE_KNN_K = 3
    VALIDATE_ECO_TRACK = True

    validate_log_path = LOGS_DIR / "ldm_validate.log"
    validate_cmd = [
        sys.executable,
        str(UTILITY_DIR / "generate_ldm.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--mode", "validate",
        "--n-raw", str(VALIDATE_N_RAW),
        "--n-selected", str(VALIDATE_N_SELECTED),
        "--target-label", str(VALIDATE_TARGET_LABEL),
        "--filtered-dir", str(ACTIVE_G06_FILTERED_DIR),
        "--balanced-seed", str(VALIDATE_BALANCED_SEED),
        "--inception-batch", str(VALIDATE_INCEPTION_BATCH),
        "--is-splits", str(VALIDATE_IS_SPLITS),
        "--knn-k", str(VALIDATE_KNN_K),
        "--results-stage-name", RESULTS_STAGE_NAME,
    ]
    if VALIDATE_ECO_TRACK:
        validate_cmd.append("--eco-track")

    add_generation_parallel_args(validate_cmd)
    env = os.environ.copy()
    print("Comando validate:")
    print(" ".join(validate_cmd))
    print("Log:", validate_log_path)

    with open(validate_log_path, "w", encoding="utf-8") as log_file:
        proc_validate = subprocess.Popen(
            validate_cmd,
            cwd=str(PROJECT_ROOT),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )

    pid_validate = proc_validate.pid
    print(f"Validate avviato - PID {pid_validate}")

    with open(validate_log_path, "r", encoding="utf-8", errors="replace") as log_file:
        while proc_validate.poll() is None:
            line = log_file.readline()
            if line:
                print(line, end="", flush=True)
            else:
                time.sleep(0.5)
        for line in log_file:
            print(line, end="", flush=True)

    print("Processo validate terminato con return code:", proc_validate.returncode)
    if proc_validate.returncode != 0:
        raise RuntimeError(f"Validate fallito. Controlla il log: {validate_log_path}")

### 8.4 Visualization of the G06 pool/filter trade-off

This read-only code converts the validation comparison into FID, before/after, PRDC-radar, and direction-aware percentage-change plots. The figures are stored under `results/2_diffusers/06_ldm_extra1361_fromscratch/plots/` and do not rerun generation.

These panels quantify the representation change induced by filtering the expanded pool. They must not be used to imply an architectural or training improvement over G05, because the underlying learned weights are the same.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

raw_vs_filtered_candidates = [
    RESULTS_METRICS_DIR / "positive" / "raw_vs_filtered_validation.csv",
    RESULTS_METRICS_DIR / "raw_vs_filtered_validation.csv",  # retained compatibility metric
]
raw_vs_filtered_path = next((path for path in raw_vs_filtered_candidates if path.exists()), None)
assert raw_vs_filtered_path is not None, (
    f"File non trovato in nessuno dei percorsi attesi: {raw_vs_filtered_candidates}. "
    "Esegui prima la cella di validate."
)

df_rvf = pd.read_csv(raw_vs_filtered_path).set_index("dataset")
balanced_name = next(name for name in df_rvf.index if name.startswith("raw_balanced"))

# 1. FID: raw completo vs filtrate
fid_order = ["raw_complete", "filtered"]
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(fid_order, df_rvf.loc[fid_order, "FID"], color=["#777777", "#2e8b57"])
ax.set_title("FID sul validation set - raw completo vs filtrate")
ax.set_ylabel("FID")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
output_path = RESULTS_PLOTS_DIR / "raw_vs_filtered_fid.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()
print("Salvato:", output_path)

raw_row = df_rvf.loc[balanced_name]
filtered_row = df_rvf.loc["filtered"]

def metric_pair(metric):
    return float(raw_row[metric]), float(filtered_row[metric])

# 2. Prima/dopo: ogni metrica mantiene la propria scala
metric_specs = [
    ("FID", "FID - più basso è meglio"),
    ("IS_mean", "Inception Score - più alto indica maggiore varietà"),
    ("precision", "Precision PRDC - più alto è meglio"),
    ("recall", "Recall PRDC - più alto indica maggiore copertura"),
]
fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)
for axis, (metric, title) in zip(axes.flat, metric_specs):
    raw_value, filtered_value = metric_pair(metric)
    delta = (filtered_value - raw_value) / raw_value * 100
    axis.plot([0, 1], [raw_value, filtered_value], color="#1f77b4", marker="o", linewidth=2.2, markersize=7)
    axis.annotate(
        "",
        xy=(1, filtered_value),
        xytext=(0, raw_value),
        arrowprops={"arrowstyle": "-|>", "color": "#1f77b4", "linewidth": 2.2, "mutation_scale": 13},
    )
    axis.annotate(
        f"{delta:+.1f}%",
        xy=(1, filtered_value),
        xytext=(7, 0),
        textcoords="offset points",
        va="center",
        fontsize=9,
        color="#1f77b4",
    )
    axis.set_xticks([0, 1], ["RAW (bilanciato)", "Filtrate"])
    axis.set_title(title)
    axis.grid(axis="y", alpha=0.25)
fig.suptitle("Effetto del filtro adattivo sull'LDM - classe positiva", fontsize=15)
output_path = RESULTS_PLOTS_DIR / "filter_before_after.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()
print("Salvato:", output_path)

# 3. Radar PRDC
prdc_metrics = ["precision", "recall", "density", "coverage"]
radar_angles = np.linspace(0, 2 * np.pi, len(prdc_metrics), endpoint=False).tolist()
radar_angles += radar_angles[:1]
raw_values = [float(raw_row[metric]) for metric in prdc_metrics]
filtered_values = [float(filtered_row[metric]) for metric in prdc_metrics]
radial_limit = min(0.5, max(raw_values + filtered_values) * 1.15)

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True}, constrained_layout=True)
for label, values, color in [("RAW (bilanciato)", raw_values, "#777777"), ("Filtrate", filtered_values, "#2e8b57")]:
    closed_values = values + values[:1]
    ax.plot(radar_angles, closed_values, color=color, linewidth=2, label=label)
    ax.fill(radar_angles, closed_values, color=color, alpha=0.18)
ax.set_xticks(radar_angles[:-1], [metric.capitalize() for metric in prdc_metrics])
ax.set_ylim(0, radial_limit)
ax.set_yticks(np.round(np.linspace(0, radial_limit, 6), 2))
ax.tick_params(axis="y", labelsize=8)
ax.set_title("Profilo PRDC prima e dopo il filtro adattivo - LDM classe positiva", pad=20)
ax.legend(loc="lower right", bbox_to_anchor=(1.25, -0.05))
output_path = RESULTS_PLOTS_DIR / "filter_prdc_radar.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()
print("Salvato:", output_path)

# 4. Delta percentuale orientato al miglioramento
delta_metrics = ["FID", "IS_mean", "precision", "recall", "density", "coverage"]
oriented_deltas = []
for metric in delta_metrics:
    raw_value, filtered_value = metric_pair(metric)
    raw_delta = (filtered_value - raw_value) / raw_value * 100
    oriented_deltas.append(-raw_delta if metric == "FID" else raw_delta)

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
y_positions = np.arange(len(delta_metrics))
colors = ["#2e8b57" if value >= 0 else "#c0392b" for value in oriented_deltas]
bars = ax.barh(y_positions, oriented_deltas, color=colors, edgecolor="white")

# Posiziona ogni label sempre verso l'esterno della barra (lontano dall'asse y),
# indipendentemente dal segno, cosi non si sovrappone mai ai tick label
for bar, value in zip(bars, oriented_deltas):
    width = bar.get_width()
    label = f"{value:+.1f}%"
    if width >= 0:
        x_pos = width
        ha = "left"
    else:
        x_pos = width
        ha = "right"
    ax.annotate(
        label,
        xy=(x_pos, bar.get_y() + bar.get_height() / 2),
        xytext=(3 if width >= 0 else -3, 0),
        textcoords="offset points",
        va="center",
        ha=ha,
        fontsize=9,
    )

# Espande i limiti dell'asse x per fare spazio alle label, evitando che vengano tagliate
x_min, x_max = ax.get_xlim()
margin = (x_max - x_min) * 0.08
ax.set_xlim(x_min - margin, x_max + margin)

ax.axvline(0, color="black", linewidth=1)
ax.set_yticks(y_positions, ["FID (segno invertito)", "IS_mean", "Precision", "Recall", "Density", "Coverage"])
ax.set_xlabel("Variazione percentuale orientata al miglioramento")
ax.set_title("Delta del filtro adattivo - LDM classe positiva")
ax.grid(axis="x", alpha=0.25)
ax.legend(
    handles=[
        Patch(facecolor="#2e8b57", label="Miglioramento"),
        Patch(facecolor="#c0392b", label="Riduzione / trade-off"),
    ],
    loc="best",
)
output_path = RESULTS_PLOTS_DIR / "filter_oriented_percent_delta.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()
print("Salvato:", output_path)

## 10. Inert manual-termination control

The stop cell is intentionally disabled so Run All cannot send a termination signal. It creates no artifact and prints only its inactive status. Any manual interruption should target a verified subprocess and rely on the existing checkpoint/resume mechanisms rather than deleting partial state.


In [ ]:
# Cella lasciata inattiva per consentire Run All senza fermare il training.
# Per fermare manualmente un processo, decommenta le righe sotto e imposta il PID giusto.
# import os
# import signal
# os.kill(pid_ldm, signal.SIGTERM)
print("Stop manuale disattivato per Run All.")

## 12. G06 artifact inventory

The final cell lists files and sizes below the G06 experiment directory and displays the first 80 entries. It supports hand-off checks for logs, pool artifacts, evaluation products, and manifests while keeping the shared G05 checkpoints outside the G06 directory by design.

Presence and byte size are not content validation. Archive documentation must pair this inventory with the G06 runtime/generation manifests and the referenced G05 checkpoint signature so the shared model identity cannot be lost.


In [ ]:
from pathlib import Path

def human_size(n_bytes):
    n = float(n_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024 or unit == "TB":
            return f"{n:.1f} {unit}"
        n /= 1024

rows = []
for path in sorted(EXPERIMENT_DIR.rglob("*")):
    if path.is_file():
        rows.append(
            {
                "path": path.relative_to(EXPERIMENT_DIR).as_posix(),
                "size": human_size(path.stat().st_size),
            }
        )

summary_df = pd.DataFrame(rows)
print("File artefatti:", len(summary_df))
display(summary_df.head(80))